# Ejercicio 4: Modelo Probabilístico

### Leandro Bravo

## Objetivo de la práctica
- Comprender los componentes del modelo vectorial mediante cálculos manuales y observación directa.
- Aplicar el modelo de espacio vectorial con TF-IDF para recuperar documentos relevantes.
- Comparar la recuperación con BM25 frente a TF-IDF.
- Analizar visualmente las diferencias entre los modelos.
- Evaluar si los rankings generados son consistentes con lo que considerarías documentos relevantes.

## Parte 0: Carga del Corpus

Utilizaremos el corpus `Gutenberg 1000`

In [7]:
import os
path = os.path.dirname(r'C:\\Users\\leand\\Desktop\\ir2026\\ir26a-main\\data\\gutenberg\\data\\')
corpus_dir = os.path.join(path, 'corpus_gutenberg_1000.txt')

with open(corpus_dir, 'r', encoding='utf-8') as f:
    corpus = f.read()
print(corpus[:500])  

The Project Gutenberg eBook of The Complete Works of William Shakespeare        This eBook is for the use of anyone anywhere in the United States and  most other parts of the world at no cost and with almost no restrictions  whatsoever. You may copy it, give it away or re-use it under the terms  of the Project Gutenberg License included with this eBook or online  at www.gutenberg.org. If you are not located in the United States,  you will have to check the laws of the country where you are loca


## Parte 1: Cálculo de TF, DF, IDF y TF-IDF

### Actividad 
1. Construye la matriz de términos (TF), y calcula la frecuencia de documentos (DF)
2. Calcula TF-IDF utilizando sklearn.
3. Visualiza los valores en un DataFrame para analizar las diferencias entre los términos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
import numpy as np


# Parse the corpus into documents
documents = corpus.strip().split('\n\n')
documents = [doc.strip() for doc in documents if doc.strip()]

print(f"Number of documents: {len(documents)}")
print(f"First document:\n{documents[0][:200]}...\n")

# 1. Build TF matrix using CountVectorizer
cv = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
tf_matrix = cv.fit_transform(documents)
tf_df = pd.DataFrame(tf_matrix.toarray(), columns=cv.get_feature_names_out())

print("TF Matrix (first 5 documents, first 10 terms):")
print(tf_df.iloc[:5, :10])

# Calculate DF (Document Frequency)
df = np.diff(tf_matrix.tocsc().indptr)  # Number of documents containing each term
df_dict = dict(zip(cv.get_feature_names_out(), df))

print(f"\nDocument Frequency for top 10 terms:")
print(sorted(df_dict.items(), key=lambda x: x[1], reverse=True)[:10])

# 2. Calculate TF-IDF using TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

print("\nTF-IDF Matrix (first 5 documents, first 10 terms):")
print(tfidf_df.iloc[:5, :10])

# 3. Visualize term statistics
term_stats = pd.DataFrame({
    'Term': cv.get_feature_names_out(),
    'DF': df,
    'Avg_TF': tf_df.mean(),
    'Avg_TFIDF': tfidf_df.mean()
})
term_stats = term_stats.sort_values('Avg_TFIDF', ascending=False)

print("\nTop 15 terms by average TF-IDF:")
print(term_stats.head(15))

Number of documents: 1
First document:
﻿The Project Gutenberg eBook of The Complete Works of William Shakespeare        This eBook is for the use of anyone anywhere in the United States and  most other parts of the world at no cost and wit...

TF Matrix (first 5 documents, first 10 terms):
    away  called   came     ce   come  country   dans    day  death    den
0  40614   35240  50549  29186  62463    28206  30385  62276  30216  31188

Document Frequency for top 10 terms:
[('away', np.int32(1)), ('called', np.int32(1)), ('came', np.int32(1)), ('ce', np.int32(1)), ('come', np.int32(1)), ('country', np.int32(1)), ('dans', np.int32(1)), ('day', np.int32(1)), ('death', np.int32(1)), ('den', np.int32(1))]


## Parte 2: Ranking de documentos usando TF-IDF

### Actividad 

1. Dada una consulta, construye el vector de consulta
2. Calcula la similitud coseno entre la consulta y cada documento usando los vectores TF-IDF
3. Genera un ranking de los documentos ordenados por relevancia.
4. Muestra los resultados en una tabla.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define a query
query = "king love death"

# 2. Transform query using the same TF-IDF vectorizer
query_vector = tfidf_vectorizer.transform([query])

# 3. Calculate cosine similarity between query and all documents
similarities = cosine_similarity(query_vector, tfidf_matrix)[0]

# 4. Create ranking dataframe
ranking_df = pd.DataFrame({
    'Document_ID': range(len(documents)),
    'Similarity_Score': similarities,
    'Document_Preview': [doc[:100] + '...' for doc in documents]
})

# Sort by similarity score in descending order
ranking_df = ranking_df.sort_values('Similarity_Score', ascending=False).reset_index(drop=True)
ranking_df['Rank'] = range(1, len(ranking_df) + 1)

print(f"Query: '{query}'\n")
print("Top 10 relevant documents (TF-IDF):")
print(ranking_df[['Rank', 'Document_ID', 'Similarity_Score', 'Document_Preview']].head(10))

Query: 'king love death'

Top 10 relevant documents (TF-IDF):
   Rank  Document_ID  Similarity_Score  \
0     1            0          0.092793   

                                    Document_Preview  
0  ﻿The Project Gutenberg eBook of The Complete W...  


## Parte 3: Ranking con BM25

### Actividad 

1. Implementa un sistema de recuperación usando el modelo BM25.
2. Usa la misma consulta del ejercicio anterior.
3. Calcula el score BM25 para cada documento y genera un ranking.
4. Compara manualmente con el ranking de TF-IDF.

In [ ]:
from collections import Counter
import math

# BM25 parameters
k1 = 1.5  
b = 0.75  

tokenized_docs = [doc.lower().split() for doc in documents]

doc_lengths = [len(doc) for doc in tokenized_docs]
avg_doc_length = sum(doc_lengths) / len(doc_lengths)

# 3. Calculate IDF for BM25
N = len(documents)  
idf_bm25 = {}

for term in cv.get_feature_names_out():
    doc_freq = sum(1 for doc in tokenized_docs if term in doc)
    # BM25 IDF formula
    idf_bm25[term] = math.log((N - doc_freq + 0.5) / (doc_freq + 0.5) + 1)

query_terms = query.lower().split()

# 5. Calculate BM25 scores for each document
bm25_scores = []

for doc_idx, doc_tokens in enumerate(tokenized_docs):
    score = 0
    doc_length = doc_lengths[doc_idx]
    term_freqs = Counter(doc_tokens)
    
    for term in query_terms:
        if term in idf_bm25:
            tf = term_freqs.get(term, 0)
            idf = idf_bm25[term]
            # BM25 formula
            score += idf * ((tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_length / avg_doc_length))))
    
    bm25_scores.append(score)

# 6. Create ranking dataframe for BM25
bm25_ranking_df = pd.DataFrame({
    'Document_ID': range(len(documents)),
    'BM25_Score': bm25_scores,
    'Document_Preview': [doc[:100] + '...' for doc in documents]
})

# Sort by BM25 score in descending order
bm25_ranking_df = bm25_ranking_df.sort_values('BM25_Score', ascending=False).reset_index(drop=True)
bm25_ranking_df['Rank'] = range(1, len(bm25_ranking_df) + 1)

print(f"Query: '{query}'\n")
print("Top 10 relevant documents (BM25):")
print(bm25_ranking_df[['Rank', 'Document_ID', 'BM25_Score', 'Document_Preview']].head(10))

# 7. Comparison with TF-IDF
print("\n" + "="*80)
print("COMPARISON: TF-IDF vs BM25")
print("="*80)
comparison_df = ranking_df[['Rank', 'Document_ID', 'Similarity_Score']].head(10).copy()
comparison_df.columns = ['Rank_TFIDF', 'Document_ID', 'TF-IDF_Score']
comparison_df['BM25_Rank'] = comparison_df['Document_ID'].map(
    bm25_ranking_df.reset_index().set_index('Document_ID')['Rank']
)
comparison_df['BM25_Score'] = comparison_df['Document_ID'].map(
    bm25_ranking_df.reset_index().set_index('Document_ID')['BM25_Score']
)

print(comparison_df)

Query: 'king love death'

Top 10 relevant documents (BM25):
   Rank  Document_ID  BM25_Score  \
0     1            0    2.157433   

                                    Document_Preview  
0  ﻿The Project Gutenberg eBook of The Complete W...  

COMPARISON: TF-IDF vs BM25
   Rank_TFIDF  Document_ID  TF-IDF_Score  BM25_Rank  BM25_Score
0           1            0      0.092793          1    2.157433


### Algoritmo BM25:

1. Tokenización
2. Calcular longitudes de documentos y promedio de longitud
3. Calcular TF (frecuencia de término por documento)
4. Calcular DF (document frequency) e IDF de BM25
5. Definir los parámetros de BM25 (k1, b)
6. Implementar la función de score BM25 para un documento
7. Función para calcular la probabilidad para todos los documentos de la colección a partir de una query
8. Recuperar los documentos mejor puntuados (ranking)

In [ ]:
import re

# 1. Tokenización
def tokenize(text):
    return re.findall(r'\w+', text.lower())

# 2. Calcular longitudes de documentos y promedio de longitud
def build_bm25_index(documents, k1=1.5, b=0.75):
    tokenized_docs = [tokenize(doc) for doc in documents]
    doc_lengths = [len(tokens) for tokens in tokenized_docs]
    avg_doc_length = sum(doc_lengths) / len(doc_lengths)
    N = len(documents)

    # 3. Calcular TF (frecuencia de término por documento)
    term_freqs = [Counter(tokens) for tokens in tokenized_docs]

    # 4. Calcular DF e IDF de BM25
    df_counts = Counter()
    for tokens in tokenized_docs:
        df_counts.update(set(tokens))

    idf_bm25 = {
        term: math.log((N - df + 0.5) / (df + 0.5) + 1)
        for term, df in df_counts.items()
    }

    return {
        'tokenized_docs': tokenized_docs,
        'doc_lengths': doc_lengths,
        'avg_doc_length': avg_doc_length,
        'term_freqs': term_freqs,
        'idf_bm25': idf_bm25,
        'k1': k1,
        'b': b,
        'N': N
    }

# 6. Implementar la función de score BM25 para un documento
def bm25_score_doc(query_terms, doc_idx, index):
    score = 0.0
    freqs = index['term_freqs'][doc_idx]
    doc_length = index['doc_lengths'][doc_idx]
    denom_factor = index['k1'] * (1 - index['b'] + index['b'] * doc_length / index['avg_doc_length'])

    for term in query_terms:
        if term not in index['idf_bm25']:
            continue
        tf = freqs.get(term, 0)
        if tf == 0:
            continue
        score += index['idf_bm25'][term] * ((tf * (index['k1'] + 1)) / (tf + denom_factor))
    return score

# 7. Función para calcular la probabilidad para todos los documentos de la colección a partir de una query
def bm25_rank(query, documents, index, top_k=None):
    query_terms = tokenize(query)
    scores = [
        bm25_score_doc(query_terms, doc_idx, index)
        for doc_idx in range(index['N'])
    ]

    ranking_df = pd.DataFrame({
        'Document_ID': range(index['N']),
        'BM25_Score': scores,
        'Document_Preview': [doc[:100] + '...' for doc in documents]
    })

    ranking_df = ranking_df.sort_values('BM25_Score', ascending=False).reset_index(drop=True)
    ranking_df['Rank'] = range(1, len(ranking_df) + 1)

    if top_k:
        return ranking_df.head(top_k)
    return ranking_df

# 5. Definir los parámetros de BM25 (k1, b) y crear índice
bm25_index = build_bm25_index(documents, k1=1.5, b=0.75)

# 8. Recuperar los documentos mejor puntuados (ranking)
bm25_ranking = bm25_rank(query, documents, bm25_index, top_k=10)

print(f"Query: '{query}'")
print(f"N documents: {bm25_index['N']}")
print(f"Average document length: {bm25_index['avg_doc_length']:.2f}\n")
print(bm25_ranking[['Rank', 'Document_ID', 'BM25_Score', 'Document_Preview']])

Query: 'king love death'
N documents: 1
Average document length: 81325945.00

   Rank  Document_ID  BM25_Score  \
0     1            0    2.157511   

                                    Document_Preview  
0  ﻿The Project Gutenberg eBook of The Complete W...  


## Parte 4: Comparación visual entre TF-IDF y BM25

### Actividad 

1. Utiliza un gráfico de barras para visualizar los scores obtenidos por cada documento según TF-IDF y BM25.
2. Compara los rankings visualmente.
3. ¿Qué documentos obtienen scores más altos en un modelo que en otro?
4. ¿A qué se podría deber esta diferencia?

In [ ]:
import matplotlib as plt

# Merge TF-IDF and BM25 scores for the same documents
comparison_df = pd.merge(
    ranking_df[['Document_ID', 'Similarity_Score', 'Rank']],
    bm25_ranking_df[['Document_ID', 'BM25_Score', 'Rank']],
    on='Document_ID',
    suffixes=('_TFIDF', '_BM25')
)

comparison_df['Score_Diff'] = comparison_df['BM25_Score'] - comparison_df['Similarity_Score']
comparison_df = comparison_df.sort_values('BM25_Score', ascending=False).reset_index(drop=True)

# Plot the scores
fig, ax = plt.subplots(figsize=(10, 5))
comparison_df.plot(
    x='Document_ID',
    y=['Similarity_Score', 'BM25_Score'],
    kind='bar',
    ax=ax,
    width=0.8
)
ax.set_title('Comparación de scores TF-IDF vs BM25 por documento')
ax.set_xlabel('Document ID')
ax.set_ylabel('Score')
ax.legend(['TF-IDF', 'BM25'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Print comparison table and rank differences
print("Comparación de documentos TF-IDF vs BM25:\n")
print(comparison_df[['Document_ID', 'Rank_TFIDF', 'Rank_BM25', 'Similarity_Score', 'BM25_Score', 'Score_Diff']])

higher_bm25 = comparison_df[comparison_df['BM25_Score'] > comparison_df['Similarity_Score']]['Document_ID'].tolist()
higher_tfidf = comparison_df[comparison_df['Similarity_Score'] > comparison_df['BM25_Score']]['Document_ID'].tolist()

print("\nDocumentos con BM25 > TF-IDF:", higher_bm25)
print("Documentos con TF-IDF > BM25:", higher_tfidf)

ModuleNotFoundError: No module named 'matplotlib'